# 🎙️ Podcast → 話者分離つき文字起こし

Apple PodcastのURLを貼るだけで、**誰が何を話したか**を区別した文字起こしテキストを生成します。

```
URL入力 → 音声DL → WhisperX（文字起こし＋話者分離） → txt出力
```

### ⚡ 最初に必ずGPUを有効にしてください
メニュー → `ランタイム` → `ランタイムのタイプを変更` → **T4 GPU**

---
## STEP 1: 環境構築（初回のみ・約3分）

In [6]:
# パッケージ修復（torchvision互換性エラー対策）
!pip install --upgrade torchvision transformers
# 全パッケージをクリーンインストール
!pip install --upgrade torch torchvision torchaudio transformers whisperx

INFO: pip is looking at multiple versions of whisperx to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of whisperx to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 10.5 MB/s eta 0:00:00
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [1]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 📦 パッケージインストール
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# whisperX = Whisper + 話者分離(pyannote) + 単語アライメント
!pip install -q git+https://github.com/m-bain/whisperX.git
!pip install -q feedparser requests

# GPU確認
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU未検出 → CPUで動作します（処理は遅くなります）")

print("✅ インストール完了！")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 6.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 112.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 893.7/893.7 kB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

---
## STEP 2: Hugging Face トークン設定

話者分離に使う pyannote モデルにはHugging Faceのトークンが必要です（無料）。

### 初回のみの準備:
1. https://huggingface.co/settings/tokens でアカウント作成 → トークン発行
2. 以下2つのモデルページで **"Agree"** をクリック（利用規約の同意）:
   - https://huggingface.co/pyannote/segmentation-3.0
   - https://huggingface.co/pyannote/speaker-diarization-3.1
3. Colabの 🔑（左サイドバー）→ `HF_TOKEN` として保存

In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 🔑 Hugging Face トークン設定
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os

# 方法A: Colabシークレットから
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("✅ シークレットからHFトークンを読み込みました")
except Exception:
    pass

# 方法B: 手入力
if not os.environ.get("HF_TOKEN"):
    from getpass import getpass
    os.environ["HF_TOKEN"] = getpass("Hugging Face トークンを入力: ")

HF_TOKEN = os.environ["HF_TOKEN"]
print(f"   トークン: {HF_TOKEN[:8]}...{HF_TOKEN[-4:]}")

Hugging Face トークンを入力: ··········
   トークン: hf_BtLmW...SUNW


---
## STEP 3: 設定（ここだけ毎回編集）

In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 📝 ここを編集してください
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── 入力方法を選んでください ──

# 方法A: Apple PodcastのURLを貼る
PODCAST_URL = "https://podcasts.apple.com/jp/podcast/%E3%83%87%E3%83%87%E3%83%87%E3%83%BC%E3%82%BF-%E3%81%82%E3%81%8D%E3%81%AA%E3%81%84-%E3%83%87%E3%83%BC%E3%82%BF%E3%81%AE%E8%A9%B1/id1743291335?i=1000664413513"  # ← URLを貼り付け

# 方法B: 音声ファイルを直接アップロードする場合は空欄のままで、
#        STEP 4 実行時にアップロードダイアログが出ます

# ── Whisperモデル ──
# large-v3 が最も高精度（日本語に強い）。VRAM不足なら medium に変更
WHISPER_MODEL = "large-v3"

# ── 言語 ──
LANGUAGE = "ja"

# ── 話者数（わかっている場合） ──
# None = 自動検出 / 数値指定で精度向上（例: 2人の対談なら 2）
NUM_SPEAKERS = None  # 例: 2

print("✅ 設定OK")
print(f"   入力: {'URL指定' if PODCAST_URL else 'ファイルアップロード'}")
print(f"   モデル: {WHISPER_MODEL}")
print(f"   話者数: {'自動検出' if NUM_SPEAKERS is None else f'{NUM_SPEAKERS}人'}")

✅ 設定OK
   入力: URL指定
   モデル: large-v3
   話者数: 自動検出


---
## STEP 4: 実行 ▶

所要時間の目安（T4 GPU + large-v3）:
- 30分の音声 → 約3〜5分
- 1時間の音声 → 約6〜12分

In [5]:
# パッケージ修復（torchvision互換性エラー対策）
!pip install --upgrade torchvision transformers

  Using cached transformers-5.2.0-py3-none-any.whl.metadata (32 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 53.8 MB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 3.4.0
    Uninstalling triton-3.4.0:
      Successfully uninstalled triton-3.4.0
  Attempting uninstall: nvidia-nccl-cu12
    Found existing installation: nvidia-nccl-cu12 2.27.3
    Uninstalling nvidia-nccl-cu12-2.27.3:
      Successfully uninstalled nvidia-nccl-cu12-2.27.3
  Attempting uninstall: torch
    Found existing installation: torch 2.8.0
    Uninstalling torch-2.8.0:
      Successfully uninstalled torch-2.8.0
  Attempting uninstall: huggingface-h

In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 🚀 実行
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import re
import time
import requests
import feedparser
import whisperx
import torch
from pathlib import Path
from datetime import datetime

OUTPUT_DIR = Path("/content/output")
OUTPUT_DIR.mkdir(exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"
start_time = time.time()

episode_title = "podcast_episode"
audio_path = None

print("=" * 60)
print("  🎙️ 話者分離つき文字起こし 開始")
print("=" * 60)
print()

# ──────────────────────────────────────
# 1. 音声ファイルの取得
# ──────────────────────────────────────
if PODCAST_URL:
    # Apple Podcast URL → RSS → 音声URL
    podcast_id_match = re.search(r'/id(\d+)', PODCAST_URL)
    if not podcast_id_match:
        raise ValueError("URLからポッドキャストIDを抽出できません")
    podcast_id = podcast_id_match.group(1)
    ep_id_match = re.search(r'[?&]i=(\d+)', PODCAST_URL)
    episode_id = ep_id_match.group(1) if ep_id_match else None

    print("📡 iTunes APIでRSSフィードを取得中...")
    itunes = requests.get(
        f"https://itunes.apple.com/lookup?id={podcast_id}&entity=podcast", timeout=30
    ).json()
    if itunes.get("resultCount", 0) == 0:
        raise ValueError("ポッドキャストが見つかりません")

    feed_url = itunes["results"][0].get("feedUrl")
    podcast_name = itunes["results"][0].get("collectionName", "")
    print(f"   ポッドキャスト: {podcast_name}")

    feed = feedparser.parse(feed_url)
    target = None
    if episode_id:
        for entry in feed.entries:
            if episode_id in (getattr(entry, 'id', '') or ''):
                target = entry
                break
    if not target:
        target = feed.entries[0]

    episode_title = getattr(target, 'title', 'episode')
    audio_url = None
    for link in getattr(target, 'enclosures', []):
        if 'audio' in link.get('type', ''):
            audio_url = link['href']
            break
    if not audio_url:
        for link in getattr(target, 'links', []):
            href = link.get('href', '')
            if any(ext in href.lower() for ext in ['.mp3', '.m4a', '.wav']):
                audio_url = href
                break
    if not audio_url:
        raise ValueError("音声URLが見つかりません。ファイルを直接アップロードしてください。")

    print(f"   エピソード: {episode_title}")
    print(f"⬇️ 音声ダウンロード中...")
    fname = audio_url.split('/')[-1].split('?')[0]
    if not any(fname.endswith(e) for e in ['.mp3','.m4a','.wav','.aac']):
        fname += '.mp3'
    audio_path = str(OUTPUT_DIR / fname)
    r = requests.get(audio_url, stream=True, timeout=120)
    r.raise_for_status()
    with open(audio_path, 'wb') as f:
        for chunk in r.iter_content(8192):
            f.write(chunk)
    size_mb = Path(audio_path).stat().st_size / 1024 / 1024
    print(f"✅ ダウンロード完了 ({size_mb:.1f}MB)")

else:
    # ファイルアップロード
    from google.colab import files as colab_files
    print("📁 音声ファイルをアップロードしてください:")
    uploaded = colab_files.upload()
    uploaded_name = list(uploaded.keys())[0]
    audio_path = f"/content/{uploaded_name}"
    episode_title = Path(uploaded_name).stem
    print(f"✅ アップロード完了: {uploaded_name}")

print()

# ──────────────────────────────────────
# 2. WhisperX: 文字起こし
# ──────────────────────────────────────
print(f"🎙️ WhisperX ({WHISPER_MODEL}) で文字起こし中...")
t1 = time.time()

model = whisperx.load_model(
    WHISPER_MODEL, device, compute_type=compute_type, language=LANGUAGE
)
audio = whisperx.load_audio(audio_path)
result = model.transcribe(audio, batch_size=16, language=LANGUAGE)

t_transcribe = time.time() - t1
print(f"✅ 文字起こし完了 ({t_transcribe:.0f}秒)")
print()

# ──────────────────────────────────────
# 3. WhisperX: 単語アライメント
# ──────────────────────────────────────
print("🔤 単語レベルのアライメント実行中...")
align_model, align_meta = whisperx.load_align_model(
    language_code=LANGUAGE, device=device
)
result = whisperx.align(
    result["segments"], align_model, align_meta, audio, device,
    return_char_alignments=False
)
print("✅ アライメント完了")
print()

# ──────────────────────────────────────
# 4. WhisperX: 話者分離 (diarization)
# ──────────────────────────────────────
print("👥 話者分離（ダイアライゼーション）実行中...")
t2 = time.time()

diarize_model = whisperx.DiarizationPipeline(
    use_auth_token=HF_TOKEN, device=device
)

diarize_kwargs = {}
if NUM_SPEAKERS is not None:
    diarize_kwargs["min_speakers"] = NUM_SPEAKERS
    diarize_kwargs["max_speakers"] = NUM_SPEAKERS

diarize_segments = diarize_model(audio, **diarize_kwargs)
result = whisperx.assign_word_speakers(diarize_segments, result)

t_diarize = time.time() - t2

# 話者の一覧
speakers = set()
for seg in result["segments"]:
    if "speaker" in seg:
        speakers.add(seg["speaker"])
speakers = sorted(speakers)

print(f"✅ 話者分離完了 ({t_diarize:.0f}秒)")
print(f"   検出された話者: {len(speakers)}人 → {', '.join(speakers)}")
print()

# ──────────────────────────────────────
# 5. テキスト整形 & txt保存
# ──────────────────────────────────────
def format_time(seconds):
    """秒数を MM:SS 形式に"""
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    if h > 0:
        return f"{h:d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"

lines = []
lines.append(f"{'='*60}")
lines.append(f"  文字起こし: {episode_title}")
lines.append(f"  生成日時: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
lines.append(f"  モデル: WhisperX {WHISPER_MODEL}")
lines.append(f"  検出話者数: {len(speakers)}人")
lines.append(f"{'='*60}")
lines.append("")

prev_speaker = None
for seg in result["segments"]:
    speaker = seg.get("speaker", "不明")
    start = seg.get("start", 0)
    text = seg.get("text", "").strip()
    if not text:
        continue

    # 話者が変わったら空行 + 話者名
    if speaker != prev_speaker:
        lines.append("")
        lines.append(f"[{format_time(start)}] {speaker}:")
        prev_speaker = speaker

    lines.append(f"  {text}")

transcript_text = "\n".join(lines)

# ファイル保存
safe_title = re.sub(r'[^\w\s-]', '', episode_title)[:50].strip().replace(' ', '_')
txt_filename = f"{safe_title}_transcript.txt"
txt_path = OUTPUT_DIR / txt_filename

with open(txt_path, 'w', encoding='utf-8') as f:
    f.write(transcript_text)

total_time = time.time() - start_time

print("=" * 60)
print("  ✅ 完了！")
print("=" * 60)
print(f"  ⏱️  合計: {total_time:.0f}秒 ({total_time/60:.1f}分)")
print(f"       文字起こし: {t_transcribe:.0f}秒 / 話者分離: {t_diarize:.0f}秒")
print(f"  👥 話者: {len(speakers)}人 ({', '.join(speakers)})")
print(f"  📄 出力: {txt_path}")
print(f"  📊 文字数: {len(transcript_text):,}")

  🎙️ 話者分離つき文字起こし 開始

📡 iTunes APIでRSSフィードを取得中...
   ポッドキャスト: デデデータ!!〜“あきない”データの話〜
   エピソード: 第192回「『あなたの★5は信用されていない』—— Netflixアルゴリズム進化論:いかにして『あなたが観たいもの』をあなたより正確に知るようになったか」
⬇️ 音声ダウンロード中...
✅ ダウンロード完了 (43.3MB)

🎙️ WhisperX (large-v3) で文字起こし中...


ModuleNotFoundError: Could not import module 'Pipeline'. Are this object's requirements defined correctly?

---
## STEP 5: 結果の確認 & ダウンロード

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 👀 プレビュー（先頭50行）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print(transcript_text[:3000])
print()
print("...（以下省略）")
print(f"\n📄 全文: {len(transcript_text):,}文字")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 📥 txtファイルをダウンロード
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from google.colab import files
files.download(str(txt_path))
print(f"✅ {txt_filename} をダウンロードしました")

---
## 💡 話者名を置換する（オプション）

出力の `SPEAKER_00`, `SPEAKER_01` を実際の名前に置き換えたい場合に使ってください。

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ✏️ 話者名の置換（任意）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ← 実際の名前に書き換えてからセルを実行
SPEAKER_NAMES = {
    "SPEAKER_00": "ホスト",
    "SPEAKER_01": "ゲスト",
    # "SPEAKER_02": "ゲスト2",  # 3人目がいれば追加
}

renamed_text = transcript_text
for old_name, new_name in SPEAKER_NAMES.items():
    renamed_text = renamed_text.replace(old_name, new_name)

# 保存
renamed_path = OUTPUT_DIR / f"{safe_title}_transcript_named.txt"
with open(renamed_path, 'w', encoding='utf-8') as f:
    f.write(renamed_text)

print("✅ 話者名を置換しました")
print(renamed_text[:2000])
print("\n...")

# ダウンロード
from google.colab import files
files.download(str(renamed_path))